# Notebook 2: Feature Extraction with Pretrained CNN

**Purpose**: Extract embeddings using pretrained ResNet50 and cache for fast training iterations.

This notebook:
1. Loads pretrained ResNet50 (removes final classification layer)
2. Extracts 2048-dimensional embeddings for all X-ray images
3. Caches features as `.npz` files for efficient reuse
4. Visualizes feature space using t-SNE/UMAP

In [ ]:
import torch
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import os
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## Step 1: Load Pretrained ResNet50

In [ ]:
# Load pretrained ResNet50
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Remove final classification layer to get embeddings
model = torch.nn.Sequential(*list(model.children())[:-1])  # 2048-dim embeddings
model.to(device)
model.eval()

print(f"Model loaded on {device}")
print(f"Output dimension: 2048")

## Step 2: Define Data Transform and Custom Dataset

In [ ]:
# ImageNet preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class XRayDataset(Dataset):
    """Custom dataset for loading X-ray images."""
    
    def __init__(self, image_dir, transform=None, label=None):
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.label = label
        
        # Find all image files
        self.image_files = list(self.image_dir.glob('*.png')) + list(self.image_dir.glob('*.jpg'))
        print(f"Found {len(self.image_files)} images in {image_dir}")
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, img_path.name, self.label

print("Dataset class defined")

## Step 3: Extract Features from Training Data

In [ ]:
def extract_features(dataset, model, device, batch_size=32):
    """Extract features for all images in a dataset."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for batch_images, file_names, labels in tqdm(loader, desc="Extracting features"):
            batch_images = batch_images.to(device)
            
            # Forward pass through ResNet50 (without classification layer)
            embeddings = model(batch_images)
            embeddings = embeddings.squeeze(-1).squeeze(-1)  # Remove spatial dimensions
            
            all_features.append(embeddings.cpu().numpy())
            
            # Assuming labels are consistent for all images in dataset
            if labels[0] is not None:
                all_labels.extend([labels[0].item() if isinstance(labels[0], torch.Tensor) else labels[0] 
                                  for _ in range(len(batch_images))])
    
    features = np.vstack(all_features)
    labels = np.array(all_labels) if all_labels else None
    
    return features, labels

print("Feature extraction function defined")

In [ ]:
# Create data directory paths
data_raw_dir = Path("../data")
data_features_dir = Path("../embeddings")
data_features_dir.mkdir(parents=True, exist_ok=True)

# This assumes the dataset is organized as:
# data/raw/train/ and data/raw/test/ with subdirectories for each class
# Adjust paths based on actual dataset structure

# Example for hierarchical dataset:
# train_normal_dir = data_raw_dir / "train" / "NORMAL"
# train_pneumonia_dir = data_raw_dir / "train" / "PNEUMONIA"
# test_normal_dir = data_raw_dir / "test" / "NORMAL"
# test_pneumonia_dir = data_raw_dir / "test" / "PNEUMONIA"

print(f"Raw data directory: {data_raw_dir}")
print(f"Features directory: {data_features_dir}")
print("\nNote: Adjust dataset paths below based on actual directory structure")

In [ ]:
# Extract training features
# This is a template - modify based on actual dataset organization

# Example extraction (adjust for your dataset structure):
# train_normal_dataset = XRayDataset(train_normal_dir, transform=transform, label=0)
# train_pneumonia_dataset = XRayDataset(train_pneumonia_dir, transform=transform, label=1)
# train_dataset = ConcatDataset([train_normal_dataset, train_pneumonia_dataset])

# features_train, labels_train = extract_features(train_dataset, model, device)
# print(f"Training features shape: {features_train.shape}")
# print(f"Training labels shape: {labels_train.shape}")

print("Feature extraction template - modify paths for your dataset")

## Step 4: Cache Features as NPZ Files

In [ ]:
# Save training features (uncomment once extraction is complete)
# np.savez(
#     data_features_dir / "train_features.npz",
#     features=features_train,
#     labels=labels_train
# )
# print(f"Training features saved to {data_features_dir / 'train_features.npz'}")

# Save test features
# test_normal_dataset = XRayDataset(test_normal_dir, transform=transform, label=0)
# test_pneumonia_dataset = XRayDataset(test_pneumonia_dir, transform=transform, label=1)
# test_dataset = ConcatDataset([test_normal_dataset, test_pneumonia_dataset])
# features_test, labels_test = extract_features(test_dataset, model, device)

# np.savez(
#     data_features_dir / "test_features.npz",
#     features=features_test,
#     labels=labels_test
# )
# print(f"Test features saved to {data_features_dir / 'test_features.npz'}")

print("Feature caching template")

## Step 5: Visualize Feature Space

In [ ]:
# Optional: Visualize feature space with t-SNE or UMAP
# Uncomment after features are extracted

# from sklearn.manifold import TSNE
# 
# # Sample features if dataset is large (>1000 samples)
# sample_size = min(1000, len(features_train))
# sample_indices = np.random.choice(len(features_train), sample_size, replace=False)
# features_sample = features_train[sample_indices]
# labels_sample = labels_train[sample_indices]
# 
# # t-SNE reduction to 2D
# tsne = TSNE(n_components=2, random_state=42, perplexity=30)
# features_2d = tsne.fit_transform(features_sample)
# 
# # Plot
# plt.figure(figsize=(10, 8))
# scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels_sample, cmap='viridis', alpha=0.6)
# plt.colorbar(scatter, label='Class')
# plt.title('Feature Space Visualization (t-SNE)')
# plt.xlabel('t-SNE 1')
# plt.ylabel('t-SNE 2')
# plt.show()

print("Feature visualization template")

## Summary

Features extracted and cached to `xray-shapley/embeddings/`. Ready for XGBoost training in Notebook 3.